In [ ]:
import os
import random
import time
import asyncio
import json
from typing import Any, List, Optional

from pydantic import BaseModel
from dotenv import load_dotenv

# BeeAI Framework imports
from beeai_framework.backend import ChatModel, UserMessage
from beeai_framework.errors import FrameworkError

# -----------------------------------------------------------------------------
# 1. Load environment & credentials
# -----------------------------------------------------------------------------
# This command loads variables from a file named `.env` in the same directory.
load_dotenv()

# --- IMPORTANT ---
# Create a file named `.env` in the same directory as this script.
# It should contain your Watsonx credentials like this:
#
# WATSONX_API_KEY="your_api_key_here"
# WATSONX_PROJECT_ID="your_project_id_here"
# WATSONX_API_URL="https://us-south.ml.cloud.ibm.com"
#

# FIX: Removed trailing spaces from the environment variable names.
# For example, "WATSONX_PROJECT_ID " is now "WATSONX_PROJECT_ID".
WATSONX_API_KEY    = os.getenv("WATSONX_API_KEY")
WATSONX_PROJECT_ID = os.getenv("PROJECT_ID")
WATSONX_API_URL    = os.getenv("WATSONX_URL")

# Validate credentials early to prevent errors later.
if not all([WATSONX_API_KEY, WATSONX_PROJECT_ID, WATSONX_API_URL]):
    missing = [name for name, val in [
        ("WATSONX_API_KEY", WATSONX_API_KEY),
        ("WATSONX_PROJECT_ID", WATSONX_PROJECT_ID),
        ("WATSONX_API_URL", WATSONX_API_URL),
    ] if not val]
    # This error will now clearly state which variables are missing.
    raise RuntimeError(f"Missing Watsonx credential(s): {', '.join(missing)}. Please check your .env file.")

# -----------------------------------------------------------------------------
# 2. LLM response schema
# -----------------------------------------------------------------------------
class ConflictCheck(BaseModel):
    """Defines the expected JSON structure from the LLM."""
    is_conflict: bool

# -----------------------------------------------------------------------------
# 3. Simulation primitives
# -----------------------------------------------------------------------------
class Task:
    """A simple representation of a task that may or may not need a shared resource."""
    def __init__(self, task_id: int, needs_resource: bool):
        self.id = task_id
        self.needs_resource = needs_resource
    def __repr__(self):
        return f"T{self.id}({'R' if self.needs_resource else '-'})"

class WorldState:
    """Logs events during the simulation for observability."""
    def __init__(self):
        self.log: List[str] = []
        self.add_log("World state initialized.")
    def add_log(self, msg: str):
        ts = time.strftime("%H:%M:%S")
        entry = f"[{ts}] {msg}"
        print(entry)
        self.log.append(entry)

# -----------------------------------------------------------------------------
# 4. Orchestrator with Watsonx LLM
# -----------------------------------------------------------------------------
class AutoSelfOrchestrator:
    """Manages the interaction with the Watsonx LLM."""
    def __init__(self, world: WorldState):
        self.world = world
        self.llm: Optional[ChatModel] = None
        self._init_llm()

    def _init_llm(self):
        """Initializes the Watsonx ChatModel from the beeai_framework."""
        self.world.add_log("Initializing Watsonx LLM client...")
        try:
            self.llm = ChatModel.from_name(
                "watsonx:meta-llama/llama-4-maverick-17b-128e-instruct-fp8",
                settings={
                    "api_key":    WATSONX_API_KEY,
                    "project_id": WATSONX_PROJECT_ID,
                    "api_base":   WATSONX_API_URL,
                    "temperature": 0.1,
                    "top_p":       0.9,
                }
            )
            self.world.add_log("✅ LLM initialized.")
        except FrameworkError as e:
            self.world.add_log(f"❌ LLM init failed: {e}")
            self.llm = None

    async def predict_conflict(self, t1: Task, t2: Task) -> bool:
        """Asks the LLM to predict if two tasks will conflict."""
        prompt = (
            f"Task A needs_resource={t1.needs_resource}\n"
            f"Task B needs_resource={t2.needs_resource}\n"
            "Do they conflict? A conflict occurs if both need the resource. Return JSON {\"is_conflict\": true/false}."
        )
        self.world.add_log(f"📡 [LLM VERIFY]\n{prompt}")
        
        # Fallback to a simple rule if the LLM is not available.
        if not self.llm:
            self.world.add_log("⚠️ LLM not available. Using rule-based fallback.")
            return t1.needs_resource and t2.needs_resource

        try:
            # Use the framework to call the LLM and get a structured response.
            resp = await self.llm.create_structure(
                schema=ConflictCheck,
                messages=[UserMessage(prompt)]
            )
            is_conflict = resp.object["is_conflict"]
            icon = "🚨 CONFLICT" if is_conflict else "✅ No conflict"
            self.world.add_log(f"💡 [LLM RESULT] {icon} between {t1} & {t2}")
            return is_conflict
        except Exception as e:
            self.world.add_log(f"⚠️ LLM verification error: {e}. Using rule-based fallback.")
            # Fallback to rule-based logic in case of an API or parsing error.
            return t1.needs_resource and t2.needs_resource

# -----------------------------------------------------------------------------
# 5. Core simulation loop
# -----------------------------------------------------------------------------
async def run_cycle(queue: List[Task], orchestrator: AutoSelfOrchestrator, use_orch: bool) -> int:
    """Processes one cycle of the simulation, executing one or two tasks."""
    if not queue:
        return 0
        
    if len(queue) >= 2:
        t1, t2 = queue[0], queue[1]
        
        # Determine if there's a conflict.
        conflict = False
        if use_orch:
            # Use the LLM orchestrator to check for conflicts.
            conflict = await orchestrator.predict_conflict(t1, t2)
        else:
            # Use a naive rule-based check.
            conflict = t1.needs_resource and t2.needs_resource
            if conflict:
                orchestrator.world.add_log(f"[Naive] 💥 Conflict detected between {t1} & {t2}, executing one.")

        if conflict:
            # If there's a conflict, only execute the first task.
            queue.pop(0)
            return 1
        else:
            # If no conflict, execute both tasks.
            if not use_orch:
                 orchestrator.world.add_log(f"[Naive] ✅ No conflict detected between {t1} & {t2}, executing both.")
            queue.pop(0)
            queue.pop(0)
            return 2
            
    else: # Only one task left in the queue
        orchestrator.world.add_log("[Exec] Executing final task in queue.")
        queue.pop(0)
        return 1

async def run_strategy(tasks: List[Task], orchestrator: AutoSelfOrchestrator, use_orch: bool) -> float:
    """Runs a full simulation with a given strategy (Orchestrator or Naive)."""
    name = "AutoSelf Orchestrator" if use_orch else "Naive Fallback"
    queue = tasks.copy()
    completed_tasks = 0
    total_cycles = 0
    orchestrator.world.add_log(f"--- Running Simulation: {name} ---")
    
    while queue:
        total_cycles += 1
        tasks_done_this_cycle = await run_cycle(queue, orchestrator, use_orch)
        completed_tasks += tasks_done_this_cycle
        orchestrator.world.add_log(f"[Cycle {total_cycles}] Completed {tasks_done_this_cycle} task(s). {len(queue)} remaining.")

    # Avoid division by zero if no tasks were processed.
    throughput = (completed_tasks / total_cycles) if total_cycles > 0 else 0
    print(f"🏁 {name}: {completed_tasks} tasks, {total_cycles} cycles → {throughput:.2f} tasks/cycle")
    return throughput

# -----------------------------------------------------------------------------
# 6. Experiment driver
# -----------------------------------------------------------------------------
async def main():
    """Main function to run the comparative simulations."""
    NUM_TASKS = 10
    CONFLICT_PROBABILITIES = [0.0, 0.25, 0.5, 0.75, 1.0]

    for p in CONFLICT_PROBABILITIES:
        world = WorldState()
        orch  = AutoSelfOrchestrator(world)
        
        # If LLM failed to init, we can't run the orchestrated version.
        if not orch.llm:
            world.add_log("Aborting simulation run due to LLM initialization failure.")
            break

        tasks = [Task(i, random.random() < p) for i in range(NUM_TASKS)]
        print(f"\n{'='*60}\nGenerated {NUM_TASKS} tasks with resource conflict probability p={p:.2f}:\n{tasks}\n{'='*60}")

        orch_tp  = await run_strategy(tasks, orch, use_orch=True)
        naive_tp = await run_strategy(tasks, orch, use_orch=False)

        print(f"\n--- Summary for p={p:.2f} ---")
        print(f"Orchestrator Throughput: {orch_tp:.2f} tasks/cycle")
        print(f"Naive Throughput:       {naive_tp:.2f} tasks/cycle")
        print("-" * 60)

if __name__ == "__main__":
    random.seed(42)
    # Use asyncio.run() to execute the top-level async main function.
    try:
        #asyncio.run(main())
        await main()
    except RuntimeError as e:
        print(f"\n🛑 A critical error occurred: {e}")



In [ ]:
# Simple Sim

In [4]:
import os
import random
import time
import asyncio
import json
from typing import Any, List, Dict, Tuple, Optional

import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel
from dotenv import load_dotenv

# BeeAI Framework imports
from beeai_framework.backend import ChatModel, UserMessage
from beeai_framework.errors import FrameworkError

# -----------------------------------------------------------------------------
# 1. Load environment & credentials
# -----------------------------------------------------------------------------
load_dotenv()
WATSONX_API_KEY    = os.getenv("WATSONX_API_KEY")
WATSONX_PROJECT_ID = os.getenv("PROJECT_ID")
WATSONX_API_URL    = os.getenv("WATSONX_URL")

if not all([WATSONX_API_KEY, WATSONX_PROJECT_ID, WATSONX_API_URL]):
    missing = [name for name,val in [
        ("WATSONX_API_KEY", WATSONX_API_KEY),
        ("WATSONX_PROJECT_ID", WATSONX_PROJECT_ID),
        ("WATSONX_API_URL", WATSONX_API_URL),
    ] if not val]
    raise RuntimeError(f"Missing Watsonx credential(s): {', '.join(missing)}")

# -----------------------------------------------------------------------------
# 2. LLM response schema
# -----------------------------------------------------------------------------
class ConflictCheck(BaseModel):
    is_conflict: bool

# -----------------------------------------------------------------------------
# 3. Simulation primitives
# -----------------------------------------------------------------------------
class Task:
    def __init__(self, task_id: int, needs_resource: bool):
        self.id = task_id
        self.needs_resource = needs_resource
    def __repr__(self):
        return f"T{self.id}({'R' if self.needs_resource else '-'})"

# -----------------------------------------------------------------------------
# 4. Orchestrator with Watsonx LLM
# -----------------------------------------------------------------------------
class AutoSelfOrchestrator:
    def __init__(self):
        self.log: List[str] = []
        self.llm: Optional[ChatModel] = None
        self._init_llm()

    def _log(self, msg: str):
        ts = time.strftime("%H:%M:%S")
        entry = f"[{ts}] {msg}"
        print(entry)
        self.log.append(entry)

    def _init_llm(self):
        self._log("Initializing Watsonx LLM client...")
        try:
            self.llm = ChatModel.from_name(
                "watsonx:meta-llama/llama-4-maverick-17b-128e-instruct-fp8",
                settings={
                    "api_key":     WATSONX_API_KEY,
                    "project_id":  WATSONX_PROJECT_ID,
                    "api_base":    WATSONX_API_URL,
                    "temperature": 0.1,
                    "top_p":       0.9,
                }
            )
            self._log("✅ LLM initialized.")
        except FrameworkError as e:
            self._log(f"❌ LLM init failed: {e}")
            self.llm = None

    async def predict_conflict(self, t1: Task, t2: Task) -> bool:
        prompt = (
            f"Task A needs_resource={t1.needs_resource}\n"
            f"Task B needs_resource={t2.needs_resource}\n"
            "Do they conflict? Return JSON {\"is_conflict\": true/false}."
        )
        self._log(f"📡 [LLM VERIFY]\n{prompt}")
        if not self.llm:
            return t1.needs_resource and t2.needs_resource
        try:
            resp = await self.llm.create_structure(
                schema=ConflictCheck,
                messages=[UserMessage(prompt)]
            )
            is_conflict = resp.object["is_conflict"]
            icon = "🚨 CONFLICT" if is_conflict else "✅ No conflict"
            print(f"[Orch] {icon} between {t1} & {t2}")
            return is_conflict
        except Exception as e:
            self._log(f"⚠️ Verification error: {e}")
            return t1.needs_resource and t2.needs_resource

# -----------------------------------------------------------------------------
# 5. Simulation logic
# -----------------------------------------------------------------------------
async def run_conflict_simulation(num_tasks: int, p: float, orchestrator: AutoSelfOrchestrator=None) -> Dict:
    tasks = [Task(i, random.random() < p) for i in range(num_tasks)]
    queue = tasks.copy()
    completed = 0
    cycles = 0
    conflicts = 0

    use_orch = orchestrator is not None and orchestrator.llm

    while queue:
        cycles += 1
        if len(queue) >= 2:
            t1, t2 = queue[0], queue[1]
            if use_orch:
                conflict = await orchestrator.predict_conflict(t1, t2)
                if conflict:
                    conflicts += 1
                    queue.pop(0)
                    completed += 1
                else:
                    queue.pop(0); queue.pop(0)
                    completed += 2
            else:
                if t1.needs_resource and t2.needs_resource:
                    conflicts += 1
                    queue.pop(0)
                    completed += 1
                else:
                    queue.pop(0); queue.pop(0)
                    completed += 2
        else:
            queue.pop(0)
            completed += 1

    return {
        "strategy":             "AutoSelf Orchestrator" if use_orch else "Naive (Baseline)",
        "conflict_probability": p,
        "num_tasks":            num_tasks,
        "conflicts_encountered":conflicts,
        "total_cycles":         cycles,
        "throughput":           completed / cycles
    }

# -----------------------------------------------------------------------------
# 6. Experiment Runner
# -----------------------------------------------------------------------------
class ExperimentRunner:
    def __init__(self, num_tasks: int, probabilities: List[float], output_dir: str="outputs"):
        self.num_tasks     = num_tasks
        self.probabilities = probabilities
        self.output_dir    = output_dir
        self.results: List[Dict] = []
        os.makedirs(output_dir, exist_ok=True)
        self.orchestrator = AutoSelfOrchestrator()

    async def run_all_scenarios(self):
        print("--- Initializing Experiment Runner ---")
        random.seed(42)

        if not self.orchestrator.llm:
            print("\n🔴 Critical error: Orchestrator LLM not initialized. Aborting.")
            self.save_orchestrator_log()
            return

        print("\n--- Running All Experimental Scenarios ---")
        for p in self.probabilities:
            print(f"\n--- Scenario p={p:.1f} ---")
            print("  Naive strategy...")
            baseline = await run_conflict_simulation(self.num_tasks, p, orchestrator=None)
            self.results.append(baseline)
            print("  AutoSelf strategy...")
            autoself = await run_conflict_simulation(self.num_tasks, p, orchestrator=self.orchestrator)
            self.results.append(autoself)

        print(f"\n--- All done. Results in '{self.output_dir}/' ---")
        self.generate_outputs()

    def generate_outputs(self):
        df = pd.DataFrame(self.results)
        df['conflict_probability'] = pd.to_numeric(df['conflict_probability'])
        csv_path = os.path.join(self.output_dir, "simulation_data.csv")
        df.to_csv(csv_path, index=False)
        print(f"✅ CSV saved: {csv_path}")
        self.generate_results_table(df)
        self.generate_throughput_plot(df)
        self.save_orchestrator_log()

    def generate_results_table(self, df: pd.DataFrame):
        df_pivot = df.pivot_table(index='conflict_probability',
                                  columns='strategy',
                                  values=['conflicts_encountered','total_cycles'])
        latex = "\\begin{table}[h!]\n\\centering\n"
        latex += f"\\caption{{Results for {self.num_tasks} tasks}}\n"
        latex += "\\begin{tabular}{|c|c|c|c|c|}\n\\hline\n"
        latex += "p & Baseline Conflicts & AutoSelf Conflicts & Baseline Cycles & AutoSelf Cycles \\\\\n\\hline\n"
        for p in self.probabilities:
            if p not in df_pivot.index: continue
            b_conf = int(df_pivot.loc[p,('conflicts_encountered','Naive (Baseline)')])
            a_conf = int(df_pivot.loc[p,('conflicts_encountered','AutoSelf Orchestrator')])
            b_cyc  = int(df_pivot.loc[p,('total_cycles','Naive (Baseline)')])
            a_cyc  = int(df_pivot.loc[p,('total_cycles','AutoSelf Orchestrator')])
            latex += f"{p:.1f} & {b_conf} & {a_conf} & {b_cyc} & {a_cyc} \\\\\n"
        latex += "\\hline\n\\end{tabular}\n\\end{table}"
        path = os.path.join(self.output_dir, "results_table.tex")
        with open(path,'w') as f: f.write(latex)
        print(f"✅ LaTeX table saved: {path}")

    def generate_throughput_plot(self, df: pd.DataFrame):
        fig, ax = plt.subplots(figsize=(8,6))
        for strat, style in [("Naive (Baseline)", "X--"), ("AutoSelf Orchestrator", "o-")]:
            sub = df[df['strategy']==strat]
            ax.plot(sub['conflict_probability'], sub['throughput'], style, label=strat)
        ax.set_xlabel("Conflict Probability p")
        ax.set_ylabel("Throughput (tasks/cycle)")
        ax.set_title("Throughput vs Conflict Probability")
        ax.legend()
        path = os.path.join(self.output_dir, "throughput_plot.pdf")
        fig.savefig(path, bbox_inches='tight')
        plt.close(fig)
        print(f"✅ Throughput plot saved: {path}")

    def save_orchestrator_log(self):
        path = os.path.join(self.output_dir, "orchestrator_log.txt")
        with open(path,'w') as f:
            f.write("\n".join(self.orchestrator.log))
        print(f"✅ Orchestrator log saved: {path}")

# -----------------------------------------------------------------------------
# 7. Run experiments
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    runner = ExperimentRunner(num_tasks=10,
                              probabilities=[0.0,0.2,0.4,0.6,0.8,1.0])
    #asyncio.run(runner.run_all_scenarios())
           
    await runner.run_all_scenarios()


[23:33:53] Initializing Watsonx LLM client...
[23:33:57] ✅ LLM initialized.
--- Initializing Experiment Runner ---

--- Running All Experimental Scenarios ---

--- Scenario p=0.0 ---
  Naive strategy...
  AutoSelf strategy...
[23:33:57] 📡 [LLM VERIFY]
Task A needs_resource=False
Task B needs_resource=False
Do they conflict? Return JSON {"is_conflict": true/false}.
[Orch] ✅ No conflict between T0(-) & T1(-)
[23:33:59] 📡 [LLM VERIFY]
Task A needs_resource=False
Task B needs_resource=False
Do they conflict? Return JSON {"is_conflict": true/false}.
[Orch] ✅ No conflict between T2(-) & T3(-)
[23:34:00] 📡 [LLM VERIFY]
Task A needs_resource=False
Task B needs_resource=False
Do they conflict? Return JSON {"is_conflict": true/false}.
[Orch] ✅ No conflict between T4(-) & T5(-)
[23:34:00] 📡 [LLM VERIFY]
Task A needs_resource=False
Task B needs_resource=False
Do they conflict? Return JSON {"is_conflict": true/false}.
[Orch] ✅ No conflict between T6(-) & T7(-)
[23:34:00] 📡 [LLM VERIFY]
Task A needs